# Experiment 1: GSM8K answer-emergence trajectories

This notebook measures when the correct numeric answer first becomes visible during a fixed 64-step LLaDA diffusion trajectory, and whether correctness remains stable afterward. It is intentionally limited to Experiment 1: GSM8K, deterministic inference, trajectory capture, answer extraction, and summary plots.

Run it headlessly with `EXP1_STAGE=smoke` first. After inspecting the smoke artifacts, rerun with `EXP1_STAGE=full`. Valid per-problem artifacts are reused automatically. GPU selection is controlled outside the notebook with `CUDA_VISIBLE_DEVICES`; `EXP1_NUM_GPUS` controls how many visible GPUs are used.

In [ ]:
# Configuration and imports
import datetime as dt
import decimal
import gzip
import hashlib
import json
import logging
import os
import random
import re
import time
import traceback
import uuid
from decimal import Decimal, InvalidOperation
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from accelerate import Accelerator, notebook_launcher
from datasets import load_dataset
from huggingface_hub import snapshot_download

STAGE = os.getenv("EXP1_STAGE", "smoke").strip().lower()
NUM_GPUS = int(os.getenv("EXP1_NUM_GPUS", "1"))
assert STAGE in {"smoke", "full"}, "EXP1_STAGE must be smoke or full"
assert 1 <= NUM_GPUS <= 4, "EXP1_NUM_GPUS must be between 1 and 4"

PROJECT_ROOT = Path(os.getenv("EXP1_PROJECT_ROOT", Path.cwd())).resolve()
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODEL_ID = "GSAI-ML/LLaDA-8B-Instruct"
MODEL_REVISION = "6059b30"
DATASET_ID = "gsm8k"
DATASET_CONFIG = "main"
DATASET_SPLIT = "test"
SEED = 20250814
NUM_STEPS = 64
GENERATION_LENGTH = 256
BLOCK_LENGTH = 256
TEMPERATURE = 0.0
CFG_SCALE = 0.0
REMASKING = "low_confidence"
MASK_TOKEN_ID = 126336
SMOKE_SIZE = 20
EXPECTED_TEST_SIZE = 1319
EXTRACTOR_VERSION = "1"

SCIENTIFIC_CONFIG = {
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_split": DATASET_SPLIT,
    "seed": SEED,
    "num_steps": NUM_STEPS,
    "generation_length": GENERATION_LENGTH,
    "block_length": BLOCK_LENGTH,
    "temperature": TEMPERATURE,
    "cfg_scale": CFG_SCALE,
    "remasking": REMASKING,
    "mask_token_id": MASK_TOKEN_ID,
    "extractor_version": EXTRACTOR_VERSION,
}
CONFIG_JSON = json.dumps(SCIENTIFIC_CONFIG, sort_keys=True, separators=(",", ":"))
CONFIG_HASH = hashlib.sha256(CONFIG_JSON.encode()).hexdigest()[:12]
RUN_ROOT = ARTIFACT_ROOT / "experiment1_gsm8k"
CACHE_ROOT = ARTIFACT_ROOT / "cache"
TRAJECTORY_DIR = RUN_ROOT / "trajectories"
LOG_DIR = RUN_ROOT / "logs"
FAILURE_DIR = RUN_ROOT / "failures"
TABLE_DIR = RUN_ROOT / "tables"
FIGURE_DIR = RUN_ROOT / "figures"
SUMMARY_DIR = RUN_ROOT / "summaries"
for directory in [CACHE_ROOT, TRAJECTORY_DIR, LOG_DIR, FAILURE_DIR, TABLE_DIR, FIGURE_DIR, SUMMARY_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

manifest_path = RUN_ROOT / "run_manifest.json"
manifest = {"config_hash": CONFIG_HASH, "scientific_config": SCIENTIFIC_CONFIG}
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, "Run manifest mismatch"
else:
    manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))

print(json.dumps({
    "stage": STAGE, "num_gpus": NUM_GPUS,
    "cuda_visible_devices": os.getenv("CUDA_VISIBLE_DEVICES", "not set"),
    "project_root": str(PROJECT_ROOT), "run_root": str(RUN_ROOT),
    "config_hash": CONFIG_HASH,
}, indent=2))
assert not torch.cuda.is_initialized(), "Restart the kernel before launching GPU workers"

In [ ]:
# Resolve inputs once on the parent process; this does not load the model onto a GPU.
LOCAL_MODEL_PATH = Path(snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=CACHE_ROOT / "huggingface",
))
gsm8k = load_dataset(
    DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT, revision="main",
    cache_dir=str(CACHE_ROOT / "datasets"),
)
assert len(gsm8k) == EXPECTED_TEST_SIZE, (len(gsm8k), EXPECTED_TEST_SIZE)

rng = np.random.default_rng(SEED)
SMOKE_IDS = sorted(rng.permutation(EXPECTED_TEST_SIZE)[:SMOKE_SIZE].tolist())
SELECTED_IDS = SMOKE_IDS if STAGE == "smoke" else list(range(EXPECTED_TEST_SIZE))
(RUN_ROOT / "smoke_ids.json").write_text(json.dumps(SMOKE_IDS, indent=2))
print(f"Selected {len(SELECTED_IDS)} problems for stage={STAGE}")

In [ ]:
# Answer extraction, metrics, logging, and resumable artifact helpers
GT_PATTERN = re.compile(r"####\s*([^\n]+)\s*$")
NUMBER_BODY = r"[-+]?\s*(?:\$\s*)?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?"
EXPLICIT_PATTERNS = [
    re.compile(rf"(?is)####\s*({NUMBER_BODY})"),
    re.compile(rf"(?is)(?:final\s+answer|answer\s+is|therefore|thus)\s*[:=]?\s*({NUMBER_BODY})"),
    re.compile(rf"(?is)\\boxed\{{\s*({NUMBER_BODY})\s*\}}"),
]
ANY_NUMBER = re.compile(NUMBER_BODY)
PROMPT_TEMPLATE = (
    "Solve this grade-school math problem carefully. Show concise reasoning. "
    "End with exactly: Final answer: <number>\n\nProblem: {question}"
)

def canonical_decimal(text: str) -> str:
    cleaned = text.strip().replace(",", "").replace("$", "").replace("%", "")
    value = Decimal(cleaned)
    if not value.is_finite():
        raise InvalidOperation(f"Non-finite number: {text}")
    if value == value.to_integral_value():
        return str(value.quantize(Decimal("1")))
    return format(value.normalize(), "f")

def parse_ground_truth(answer_text: str) -> str:
    match = GT_PATTERN.search(answer_text)
    if not match:
        raise ValueError("Missing GSM8K #### answer delimiter")
    return canonical_decimal(match.group(1))

def extract_answer(text: str) -> Dict[str, Optional[str]]:
    if not text or not text.strip():
        return {"status": "empty", "answer": None, "method": None}
    for pattern in EXPLICIT_PATTERNS:
        matches = list(pattern.finditer(text))
        if matches:
            raw = matches[-1].group(1)
            try:
                return {"status": "ok", "answer": canonical_decimal(re.sub(r"\s+", "", raw)), "method": "explicit"}
            except (InvalidOperation, ValueError):
                return {"status": "invalid_number", "answer": None, "method": "explicit"}
    matches = list(ANY_NUMBER.finditer(text))
    if not matches:
        return {"status": "no_number", "answer": None, "method": None}
    raw = matches[-1].group(0)
    try:
        return {"status": "ok", "answer": canonical_decimal(re.sub(r"\s+", "", raw)), "method": "last_number"}
    except (InvalidOperation, ValueError):
        return {"status": "invalid_number", "answer": None, "method": "last_number"}

def derive_metrics(steps: List[Dict[str, Any]]) -> Dict[str, Any]:
    correct_steps = [row["step"] for row in steps if row["correct"] is True]
    first = min(correct_steps) if correct_steps else None
    final_correct = steps[-1]["correct"] is True
    if first is None:
        return {
            "first_correct_step": None, "normalized_first_arrival": None,
            "ever_correct": False, "final_correct": final_correct,
            "changed_after_arrival": None, "post_arrival_fraction": None,
            "trajectory_category": "never_correct",
        }
    later = [row for row in steps if row["step"] > first]
    changed = any(row["correct"] is not True for row in later)
    if not changed:
        category = "stable_correct"
    elif final_correct:
        category = "temporary_regression"
    else:
        category = "ended_incorrect"
    return {
        "first_correct_step": first, "normalized_first_arrival": first / NUM_STEPS,
        "ever_correct": True, "final_correct": final_correct,
        "changed_after_arrival": changed,
        "post_arrival_fraction": (NUM_STEPS - first) / NUM_STEPS,
        "trajectory_category": category,
    }

def artifact_path(problem_id: int) -> Path:
    return TRAJECTORY_DIR / f"problem_{problem_id:04d}.json.gz"

def atomic_write_json_gz(path: Path, payload: Dict[str, Any]) -> None:
    temporary = path.with_name(path.name + f".tmp-{uuid.uuid4().hex}")
    with gzip.open(temporary, "wt", encoding="utf-8") as handle:
        json.dump(payload, handle, ensure_ascii=False, sort_keys=True)
    os.replace(temporary, path)

def load_artifact(problem_id: int) -> Optional[Dict[str, Any]]:
    path = artifact_path(problem_id)
    if not path.exists():
        return None
    try:
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            payload = json.load(handle)
        valid = (
            payload.get("config_hash") == CONFIG_HASH
            and payload.get("problem_id") == problem_id
            and len(payload.get("steps", [])) == NUM_STEPS
            and [row.get("step") for row in payload["steps"]] == list(range(1, NUM_STEPS + 1))
            and payload["steps"][-1].get("remaining_mask_count") == 0
        )
        return payload if valid else None
    except (OSError, EOFError, json.JSONDecodeError, KeyError, TypeError):
        return None

def make_logger(rank: int) -> logging.Logger:
    logger = logging.getLogger(f"exp1.rank{rank}")
    logger.setLevel(logging.INFO)
    logger.propagate = False
    if not logger.handlers:
        formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
        file_handler = logging.FileHandler(LOG_DIR / f"{STAGE}-rank-{rank:02d}.log")
        file_handler.setFormatter(formatter)
        stream_handler = logging.StreamHandler()
        stream_handler.setFormatter(formatter)
        logger.addHandler(file_handler)
        logger.addHandler(stream_handler)
    return logger

def append_failure(rank: int, row: Dict[str, Any]) -> None:
    path = FAILURE_DIR / f"{STAGE}-rank-{rank:02d}.jsonl"
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, sort_keys=True) + "\n")
        handle.flush()
        os.fsync(handle.fileno())

# CPU-only sanity checks.
assert parse_ground_truth("work\n#### 1,234") == "1234"
assert extract_answer("Therefore, final answer: 42")["answer"] == "42"
assert extract_answer("The answer is $1,250.")["answer"] == "1250"
assert extract_answer("No numeric answer")["answer"] is None
print("Extractor sanity checks passed")

In [ ]:
# Instrumented deterministic LLaDA generation loop
def get_num_transfer_tokens(mask_index: torch.Tensor, steps: int) -> torch.Tensor:
    mask_count = mask_index.sum(dim=1, keepdim=True)
    base = mask_count // steps
    remainder = mask_count % steps
    transfers = torch.zeros(
        mask_count.size(0), steps, dtype=torch.int64, device=mask_index.device
    ) + base
    for row in range(mask_count.size(0)):
        transfers[row, :int(remainder[row].item())] += 1
    return transfers

@torch.inference_mode()
def generate_with_snapshots(
    model, prompt_ids: torch.Tensor, attention_mask: torch.Tensor
) -> Tuple[List[torch.Tensor], List[int]]:
    batch_size, prompt_width = prompt_ids.shape
    assert batch_size == 1, "The research-safe micro-batch size is one"
    tokens = torch.full(
        (batch_size, prompt_width + GENERATION_LENGTH), MASK_TOKEN_ID,
        dtype=torch.long, device=prompt_ids.device,
    )
    tokens[:, :prompt_width] = prompt_ids
    full_attention_mask = torch.cat([
        attention_mask,
        torch.ones(
            (batch_size, GENERATION_LENGTH),
            dtype=attention_mask.dtype, device=attention_mask.device,
        ),
    ], dim=-1)
    num_blocks = GENERATION_LENGTH // BLOCK_LENGTH
    assert GENERATION_LENGTH % BLOCK_LENGTH == 0
    assert NUM_STEPS % num_blocks == 0
    steps_per_block = NUM_STEPS // num_blocks
    snapshots: List[torch.Tensor] = []
    remaining_masks: List[int] = []

    for block in range(num_blocks):
        block_start = prompt_width + block * BLOCK_LENGTH
        block_end = prompt_width + (block + 1) * BLOCK_LENGTH
        block_masks = tokens[:, block_start:block_end].eq(MASK_TOKEN_ID)
        transfers = get_num_transfer_tokens(block_masks, steps_per_block)
        for local_step in range(steps_per_block):
            mask_index = tokens.eq(MASK_TOKEN_ID)
            candidate_mask = mask_index.clone()
            candidate_mask[:, :block_start] = False
            candidate_mask[:, block_end:] = False

            logits = model(tokens, attention_mask=full_attention_mask).logits
            predicted = torch.argmax(logits, dim=-1)
            probabilities = F.softmax(logits, dim=-1)
            predicted_probability = torch.gather(
                probabilities, -1, predicted.unsqueeze(-1)
            ).squeeze(-1)
            predicted = torch.where(mask_index, predicted, tokens)
            confidence = torch.where(
                candidate_mask, predicted_probability, -torch.inf
            )
            transfer_index = torch.zeros_like(tokens, dtype=torch.bool)
            for row in range(batch_size):
                count = int(transfers[row, local_step].item())
                if count:
                    chosen = torch.topk(confidence[row], k=count).indices
                    transfer_index[row, chosen] = True
            tokens[transfer_index] = predicted[transfer_index]

            generated = tokens[:, prompt_width:].detach().cpu().clone()
            snapshots.append(generated)
            remaining_masks.append(int(generated.eq(MASK_TOKEN_ID).sum().item()))
            del logits, probabilities, predicted_probability, confidence

    assert len(snapshots) == NUM_STEPS
    assert remaining_masks[-1] == 0
    return snapshots, remaining_masks

In [ ]:
# Per-GPU worker. Each process loads one model copy and owns disjoint problem IDs.
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cudnn.benchmark = False

def make_prompt(tokenizer, question: str) -> str:
    message = {"role": "user", "content": PROMPT_TEMPLATE.format(question=question)}
    return tokenizer.apply_chat_template(
        [message], add_generation_prompt=True, tokenize=False
    )

def run_problem(model, tokenizer, row: Dict[str, Any], rank: int) -> Dict[str, Any]:
    prompt = make_prompt(tokenizer, row["question"])
    encoded = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    )
    device = next(model.parameters()).device
    prompt_ids = encoded["input_ids"].to(device)
    attention_mask = encoded["attention_mask"].to(device)
    started = time.perf_counter()
    snapshots, remaining_masks = generate_with_snapshots(
        model, prompt_ids, attention_mask
    )
    ground_truth = parse_ground_truth(row["answer"])
    step_rows = []
    for step, snapshot in enumerate(snapshots, start=1):
        decoded = tokenizer.decode(snapshot[0].tolist(), skip_special_tokens=True)
        parsed = extract_answer(decoded)
        step_rows.append({
            "step": step,
            "remaining_mask_count": remaining_masks[step - 1],
            "decoded_text": decoded,
            "extraction_status": parsed["status"],
            "extraction_method": parsed["method"],
            "extracted_answer": parsed["answer"],
            "correct": parsed["answer"] == ground_truth if parsed["status"] == "ok" else None,
        })
    return {
        "config_hash": CONFIG_HASH,
        "problem_id": row["problem_id"],
        "question": row["question"],
        "reference_solution": row["answer"],
        "ground_truth": ground_truth,
        "prompt": prompt,
        "gpu_rank": rank,
        "elapsed_seconds": time.perf_counter() - started,
        "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
        "steps": step_rows,
        "metrics": derive_metrics(step_rows),
    }

def gpu_worker(selected_ids: Sequence[int]) -> None:
    accelerator = Accelerator()
    rank = accelerator.process_index
    world_size = accelerator.num_processes
    logger = make_logger(rank)
    seed_everything(SEED)

    from transformers import AutoModel, AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        str(LOCAL_MODEL_PATH), trust_remote_code=True, local_files_only=True
    )
    tokenizer.padding_side = "left"
    assert tokenizer.pad_token_id != MASK_TOKEN_ID
    model = AutoModel.from_pretrained(
        str(LOCAL_MODEL_PATH), trust_remote_code=True, local_files_only=True,
        torch_dtype=torch.bfloat16, low_cpu_mem_usage=True,
    ).to(accelerator.device).eval()

    local_dataset = load_dataset(
        DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT, revision="main",
        cache_dir=str(CACHE_ROOT / "datasets"),
    )
    assigned = [problem_id for problem_id in selected_ids if problem_id % world_size == rank]
    pending = [problem_id for problem_id in assigned if load_artifact(problem_id) is None]
    logger.info("stage=%s assigned=%d pending=%d world_size=%d", STAGE, len(assigned), len(pending), world_size)

    for position, problem_id in enumerate(pending, start=1):
        try:
            source = local_dataset[problem_id]
            row = {
                "problem_id": problem_id,
                "question": source["question"],
                "answer": source["answer"],
            }
            payload = run_problem(model, tokenizer, row, rank)
            atomic_write_json_gz(artifact_path(problem_id), payload)
            logger.info(
                "completed=%d/%d problem_id=%d first_correct=%s final_correct=%s seconds=%.1f",
                position, len(pending), problem_id,
                payload["metrics"]["first_correct_step"],
                payload["metrics"]["final_correct"],
                payload["elapsed_seconds"],
            )
        except Exception as exc:
            logger.exception("problem_id=%d failed", problem_id)
            append_failure(rank, {
                "problem_id": problem_id,
                "error_type": type(exc).__name__,
                "error": str(exc),
                "traceback": traceback.format_exc(),
                "timestamp_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
            })
            if isinstance(exc, torch.cuda.OutOfMemoryError):
                torch.cuda.empty_cache()
    accelerator.wait_for_everyone()
    logger.info("stage=%s worker complete", STAGE)

In [ ]:
# Launch the selected stage. Valid artifacts from earlier runs are skipped.
assert not torch.cuda.is_initialized(), "Restart the kernel before notebook_launcher"
notebook_launcher(gpu_worker, args=(SELECTED_IDS,), num_processes=NUM_GPUS)

In [ ]:
# Completeness check, tables, aggregate metrics, and core plots
payloads = []
missing_ids = []
for problem_id in SELECTED_IDS:
    payload = load_artifact(problem_id)
    if payload is None:
        missing_ids.append(problem_id)
    else:
        payloads.append(payload)

if missing_ids:
    (FAILURE_DIR / f"{STAGE}-missing-ids.json").write_text(json.dumps(missing_ids, indent=2))
raise_message = f"{len(missing_ids)} problems are missing; rerun the same command to retry them"
assert not missing_ids, raise_message

problem_rows = []
step_rows = []
for payload in payloads:
    metrics = payload["metrics"]
    problem_rows.append({
        "problem_id": payload["problem_id"],
        "ground_truth": payload["ground_truth"],
        "final_extracted_answer": payload["steps"][-1]["extracted_answer"],
        **metrics,
    })
    for row in payload["steps"]:
        step_rows.append({
            "problem_id": payload["problem_id"],
            "ground_truth": payload["ground_truth"],
            **{key: row[key] for key in [
                "step", "remaining_mask_count", "extraction_status",
                "extraction_method", "extracted_answer", "correct",
            ]},
        })

problem_df = pd.DataFrame(problem_rows).sort_values("problem_id").reset_index(drop=True)
step_df = pd.DataFrame(step_rows).sort_values(["problem_id", "step"]).reset_index(drop=True)
assert len(problem_df) == len(SELECTED_IDS)
assert len(step_df) == len(SELECTED_IDS) * NUM_STEPS
problem_df.to_csv(TABLE_DIR / f"{STAGE}_problem_table.csv", index=False)
problem_df.to_parquet(TABLE_DIR / f"{STAGE}_problem_table.parquet", index=False)
step_df.to_csv(TABLE_DIR / f"{STAGE}_step_table.csv", index=False)
step_df.to_parquet(TABLE_DIR / f"{STAGE}_step_table.parquet", index=False)

ever = problem_df[problem_df["ever_correct"]].copy()
categories = problem_df["trajectory_category"].value_counts().to_dict()
summary = {
    "stage": STAGE,
    "n_problems": int(len(problem_df)),
    "final_accuracy": float(problem_df["final_correct"].mean()),
    "ever_correct_rate": float(problem_df["ever_correct"].mean()),
    "mean_first_correct_step_among_ever_correct": float(ever["first_correct_step"].mean()) if len(ever) else None,
    "median_first_correct_step_among_ever_correct": float(ever["first_correct_step"].median()) if len(ever) else None,
    "mean_post_arrival_fraction_among_ever_correct": float(ever["post_arrival_fraction"].mean()) if len(ever) else None,
    "trajectory_category_counts": {key: int(value) for key, value in categories.items()},
    "final_parse_failure_rate": float((payloads and np.mean([p["steps"][-1]["extraction_status"] != "ok" for p in payloads])) or 0.0),
}
(SUMMARY_DIR / f"{STAGE}_aggregate_metrics.json").write_text(
    json.dumps(summary, indent=2, sort_keys=True)
)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if len(ever):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(ever["first_correct_step"], bins=np.arange(0.5, NUM_STEPS + 1.5, 1))
    ax.set(title="When the correct answer first appears", xlabel="First correct diffusion step", ylabel="Problems")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / f"{STAGE}_first_correct_histogram.png", dpi=200)
    plt.close(fig)

curve = step_df.assign(correct_strict=step_df["correct"].fillna(False).astype(bool)).groupby("step")["correct_strict"].mean()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(curve.index, curve.values)
ax.set(title="Strict numeric-answer accuracy during denoising", xlabel="Diffusion step", ylabel="Accuracy", ylim=(0, 1))
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"{STAGE}_correctness_by_step.png", dpi=200)
plt.close(fig)

category_order = ["stable_correct", "temporary_regression", "ended_incorrect", "never_correct"]
category_counts = problem_df["trajectory_category"].value_counts().reindex(category_order, fill_value=0)
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(category_counts.index, category_counts.values)
ax.tick_params(axis="x", rotation=15)
ax.set(title="Answer stability after first arrival", xlabel="Trajectory category", ylabel="Problems")
fig.tight_layout()
fig.savefig(FIGURE_DIR / f"{STAGE}_trajectory_categories.png", dpi=200)
plt.close(fig)

print(json.dumps(summary, indent=2))
print("Run artifacts:", RUN_ROOT)